# Agentic AI Sycophancy: GitHub Data Collection

This notebook implements Stage 1 (high-recall candidate harvest) of a two-stage design for a taxonomy-building study of sycophancy in agentic AI systems, and constructs the sampling frame for Stage 2 (stratified screening and multi-label coding, handled separately).

The notebook:
1. verifies and canonicalises 55 target repositories (six functional categories) against the GitHub API,
2. searches public GitHub issues created between 1 January 2023 and 30 June 2026, using an eight-dimension, three-level sycophancy lexicon (core / behavioural / exploratory),
3. retrieves all results per query, splitting date windows recursively where GitHub's 1,000-result cap is exceeded, flagging near-cap queries, and logging every executed query,
4. removes duplicate issues (recording all matching terms, dimensions, and term levels per issue),
5. retrieves public issue comments for all unique candidate issues, labelling zero-comment issues,
6. constructs one structured thread text per issue with author, role, and timestamp markers, and
7. builds a seeded Stage-2 sampling frame with repository caps (max 75 per repository) and dimension floors (min 25 per dimension where available), so prolific repositories cannot dominate the coded corpus.

The resulting dataset is a keyword-enriched candidate corpus for taxonomy development and empirical mapping. It is not a prevalence sample.

API keys are not included. Add `GITHUB_TOKEN` through Google Colab Secrets before running the notebook.


In [ ]:
!pip install -q requests pandas tqdm openpyxl

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    print("No GitHub token found. Public API access may still work, but rate limits will be much lower.")
else:
    print("GitHub token loaded.")

In [ ]:
HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"


In [ ]:
# Expanded repository corpus: 55 repositories across six functional categories
# (8-10 per category). Repo paths are canonicalised in the next cell; any path
# that cannot be resolved is reported and skipped.

REPO_CATEGORIES = {
    # 1. General-purpose agent SDKs and orchestration frameworks
    "microsoft/agent-framework": "Agent SDKs and orchestration",
    "microsoft/semantic-kernel": "Agent SDKs and orchestration",
    "langchain-ai/langgraph": "Agent SDKs and orchestration",
    "openai/openai-agents-python": "Agent SDKs and orchestration",
    "google/adk-python": "Agent SDKs and orchestration",
    "langchain-ai/langchain": "Agent SDKs and orchestration",
    "Significant-Gravitas/AutoGPT": "Agent SDKs and orchestration",
    "pydantic/pydantic-ai": "Agent SDKs and orchestration",
    "agno-agi/agno": "Agent SDKs and orchestration",
    "mastra-ai/mastra": "Agent SDKs and orchestration",

    # 2. Multi-agent collaboration frameworks
    "microsoft/autogen": "Multi-agent collaboration",
    "crewAIInc/crewAI": "Multi-agent collaboration",
    "camel-ai/camel": "Multi-agent collaboration",
    "FoundationAgents/MetaGPT": "Multi-agent collaboration",
    "OpenBMB/ChatDev": "Multi-agent collaboration",
    "agentscope-ai/agentscope": "Multi-agent collaboration",
    "agentscope-ai/AgentTeams": "Multi-agent collaboration",
    "TransformerOptimus/SuperAGI": "Multi-agent collaboration",
    "VoltAgent/voltagent": "Multi-agent collaboration",

    # 3. Coding and software-engineering agents
    "OpenHands/OpenHands": "Coding and software-engineering agents",
    "cline/cline": "Coding and software-engineering agents",
    "openai/codex": "Coding and software-engineering agents",
    "google-gemini/gemini-cli": "Coding and software-engineering agents",
    "anthropics/claude-code": "Coding and software-engineering agents",
    "SWE-agent/SWE-agent": "Coding and software-engineering agents",
    "aider-ai/aider": "Coding and software-engineering agents",
    "continuedev/continue": "Coding and software-engineering agents",
    "TabbyML/tabby": "Coding and software-engineering agents",

    # 4. Browser, research, and general action agents
    "browser-use/browser-use": "Browser, research, and action agents",
    "FoundationAgents/OpenManus": "Browser, research, and action agents",
    "assafelovic/gpt-researcher": "Browser, research, and action agents",
    "langchain-ai/open_deep_research": "Browser, research, and action agents",
    "SamuelSchmidgall/AgentLaboratory": "Browser, research, and action agents",
    "Skyvern-AI/skyvern": "Browser, research, and action agents",
    "nanobrowser/nanobrowser": "Browser, research, and action agents",
    "vercel-labs/agent-browser": "Browser, research, and action agents",
    "browser-use/web-ui": "Browser, research, and action agents",
    "run-llama/llama_index": "Browser, research, and action agents",

    # 5. Memory, personalisation, and stateful agent infrastructure
    "mem0ai/mem0": "Memory and personalisation infrastructure",
    "letta-ai/letta": "Memory and personalisation infrastructure",
    "getzep/graphiti": "Memory and personalisation infrastructure",
    "langchain-ai/langmem": "Memory and personalisation infrastructure",
    "langgenius/dify": "Memory and personalisation infrastructure",
    "getzep/zep": "Memory and personalisation infrastructure",
    "topoteretes/cognee": "Memory and personalisation infrastructure",
    "supermemoryai/supermemory": "Memory and personalisation infrastructure",

    # 6. Evaluation, observability, and control systems
    "promptfoo/promptfoo": "Evaluation, observability, and control",
    "confident-ai/deepeval": "Evaluation, observability, and control",
    "langfuse/langfuse": "Evaluation, observability, and control",
    "Arize-ai/phoenix": "Evaluation, observability, and control",
    "AgentOps-AI/agentops": "Evaluation, observability, and control",
    "openai/evals": "Evaluation, observability, and control",
    "explodinggradients/ragas": "Evaluation, observability, and control",
    "guardrails-ai/guardrails": "Evaluation, observability, and control",
    "NVIDIA/NeMo-Guardrails": "Evaluation, observability, and control",
}

REPOS = list(REPO_CATEGORIES.keys())

# Revised search lexicon: eight dimensions separating primary sycophancy from
# adjacent phenomena. Each term carries a diagnostic level:
#   "core"        - high-precision, direct sycophancy vocabulary
#   "behavioural" - agentic manifestations in actions, plans, and compliance
#   "exploratory" - broader discovery terms; hits are candidates only
# Removed as too ambiguous after the pilot run: "cherry-pick" (git noise),
# standalone "defers to", standalone "over-validation", "lenient", "judge bias",
# "position bias", "verbosity bias", "reward hacking".
SYCOPHANCY_TERMS = {
    "Direct Sycophancy and Agreement": [
        ("sycophancy", "core"),
        ("sycophantic", "core"),
        ("people pleasing", "core"),
        ("pandering", "core"),
        ("yes-man", "core"),
        ("overly agreeable", "core"),
        ("too agreeable", "core"),
        ("always agrees", "core"),
        ("agrees with everything", "core"),
        ("excessive agreement", "core"),
        ("excessive praise", "core"),
        ("flattery", "core"),
        ("tells the user what they want to hear", "core"),
        ("you're absolutely right", "core")
    ],
    "Opinion Conformity and Belief Reinforcement": [
        ("mirrors the user's opinion", "behavioural"),
        ("echoes the user's view", "behavioural"),
        ("confirms the user's belief", "behavioural"),
        ("reinforces a misconception", "behavioural"),
        ("reinforces an incorrect belief", "behavioural"),
        ("accepts a false premise", "behavioural"),
        ("accepts an incorrect premise", "behavioural"),
        ("uncritically agrees", "behavioural"),
        ("blind agreement", "core"),
        ("user is always right", "core"),
        ("self-reinforcing agreement", "exploratory"),
        ("echo chamber", "exploratory")
    ],
    "Capitulation and Answer Flipping": [
        ("changes answer after pushback", "core"),
        ("changes answer when challenged", "core"),
        ("reverses a correct answer", "core"),
        ("abandons the correct answer", "core"),
        ("backs down when challenged", "core"),
        ("capitulates", "core"),
        ("apologises and changes answer", "core"),
        ("apologizes and changes answer", "core"),
        ("changes its answer to please the user", "behavioural"),
        ("retreats from a justified position", "behavioural"),
        ("gives in to user pressure", "behavioural"),
        ("second-guesses a correct answer", "behavioural")
    ],
    "Evidence and Reasoning Conformity": [
        ("cherry-picks evidence", "core"),
        ("selective evidence", "behavioural"),
        ("selective retrieval", "behavioural"),
        ("ignores contradictory evidence", "behavioural"),
        ("suppresses contrary evidence", "behavioural"),
        ("researches only one side", "behavioural"),
        ("confirms the user's conclusion", "behavioural"),
        ("one-sided evidence", "exploratory"),
        ("one-sided summary", "exploratory"),
        ("biased summary", "exploratory"),
        ("slanted summary", "exploratory"),
        ("predetermined conclusion", "exploratory")
    ],
    "Agentic Action and Plan Sycophancy": [
        ("fails to challenge", "behavioural"),
        ("does not push back", "behavioural"),
        ("blindly follows", "behavioural"),
        ("accepts a bad plan", "behavioural"),
        ("validates a bad idea", "behavioural"),
        ("uncritical compliance", "behavioural"),
        ("uncritically accepts the plan", "behavioural"),
        ("ignores user-defined constraints", "behavioural"),
        ("prioritises user preference over constraints", "behavioural"),
        ("prioritizes user preference over constraints", "behavioural"),
        ("changes plan to please the user", "behavioural"),
        ("executes an unsafe request", "behavioural"),
        ("follows an unreasonable instruction", "behavioural"),
        ("fails to question the user's objective", "behavioural"),
        ("does not flag a risky assumption", "behavioural"),
        ("continues despite contradictory evidence", "behavioural"),
        ("overweights user preference", "behavioural"),
        ("follows the user's preferred answer", "behavioural"),
        ("ignores constraints", "exploratory")
    ],
    "Inter-Agent Deference and Consensus": [
        ("agent deference", "core"),
        ("premature consensus", "core"),
        ("consensus collapse", "core"),
        ("disagreement collapse", "core"),
        ("agent agrees with supervisor", "behavioural"),
        ("agent agrees with planner", "behavioural"),
        ("uncritically accepts the planner", "behavioural"),
        ("uncritically accepts the critic", "behavioural"),
        ("blindly follows the supervisor", "behavioural"),
        ("fails to challenge another agent", "behavioural"),
        ("suppresses dissent", "behavioural"),
        ("inter-agent conformity", "exploratory"),
        ("agent herding", "exploratory"),
        ("groupthink", "exploratory"),
        ("false consensus", "exploratory"),
        ("rubber-stamps", "exploratory")
    ],
    "Personalisation and Memory Sycophancy": [
        ("memory-induced sycophancy", "core"),
        ("personalisation bias", "core"),
        ("personalization bias", "core"),
        ("over-personalisation", "core"),
        ("over-personalization", "core"),
        ("remembered belief as fact", "behavioural"),
        ("uses preference as fact", "behavioural"),
        ("preference overrides evidence", "behavioural"),
        ("preference overrides current instruction", "behavioural"),
        ("prioritises stored preference over current evidence", "behavioural"),
        ("reinforces remembered belief", "behavioural"),
        ("refuses to update a user model", "behavioural"),
        ("outdated user preference", "behavioural"),
        ("stale preference", "exploratory"),
        ("memory causes agreement", "exploratory")
    ],
    "Evaluator and Judge Conformity": [
        ("evaluator agrees with the model", "behavioural"),
        ("judge favours its own answer", "behavioural"),
        ("judge favors its own answer", "behavioural"),
        ("critic fails to challenge the answer", "behavioural"),
        ("grade inflation", "exploratory"),
        ("inflated score", "exploratory"),
        ("score inflation", "exploratory"),
        ("lenient evaluation", "exploratory"),
        ("self-preference", "exploratory"),
        ("evaluator bias", "exploratory"),
        ("positive evaluation bias", "exploratory"),
        ("overly generous judging", "exploratory")
    ]
}

n_terms = sum(len(v) for v in SYCOPHANCY_TERMS.values())
from collections import Counter
level_counts = Counter(l for v in SYCOPHANCY_TERMS.values() for _, l in v)
print(f"{len(REPOS)} repositories, {n_terms} search terms, "
      f"up to {len(REPOS) * n_terms} repo-term queries.")
print("Term levels:", dict(level_counts))


In [ ]:
# Verify and canonicalise repository names before searching.
# GitHub's search API can silently return zero results for renamed or moved
# repositories, so every path is resolved to its current canonical location here.

import requests
import time

VERIFIED_REPO_CATEGORIES = {}
problems = []

for repo, category in REPO_CATEGORIES.items():
    response = requests.get(f"https://api.github.com/repos/{repo}", headers=HEADERS)

    if response.status_code == 200:
        canonical = response.json().get("full_name", repo)
        if canonical != repo:
            print(f"Renamed: {repo} -> {canonical}")
        VERIFIED_REPO_CATEGORIES[canonical] = category
    else:
        problems.append((repo, response.status_code))
        print(f"PROBLEM: {repo} returned HTTP {response.status_code}")

    time.sleep(0.5)

REPOS = list(VERIFIED_REPO_CATEGORIES.keys())
REPO_CATEGORIES = VERIFIED_REPO_CATEGORIES

print(f"\n{len(REPOS)} repositories verified.")
if problems:
    print(f"{len(problems)} repositories could not be resolved and will be skipped: {problems}")
    print("Check these paths manually on github.com before proceeding.")


In [ ]:
# Reproducibility settings

from datetime import date

# Fixed collection window. Only issues CREATED within this window are collected.
START_DATE = date(2023, 1, 1)
END_DATE = date(2026, 6, 30)

SEARCH_SLEEP_SECONDS = 2

# GitHub search returns at most 1,000 results per query (10 pages of 100).
# Queries exceeding the cap are split recursively into smaller date windows;
# queries approaching the cap are flagged in the search log.
PER_PAGE = 100
MAX_PAGES_PER_QUERY = 10
API_RESULT_CAP = 1000
NEAR_CAP_THRESHOLD = 900

# Stage-2 sampling frame parameters (repository caps and dimension floors)
RANDOM_SEED = 42
MAX_PER_REPO = 75
MIN_PER_DIMENSION = 25

# Output filenames
CANDIDATE_CSV = "github_sycophancy_candidate_issues.csv"
CANDIDATE_EXCEL = "github_sycophancy_candidate_issues_clean.xlsx"

COMMENTS_CSV = "github_sycophancy_issue_comments.csv"
COMMENTS_EXCEL = "github_sycophancy_issue_comments.xlsx"

THREADS_CSV = "github_sycophancy_issue_threads.csv"
THREADS_EXCEL = "github_sycophancy_issue_threads.xlsx"

SAMPLING_FRAME_CSV = "github_sycophancy_sampling_frame.csv"
SAMPLING_FRAME_EXCEL = "github_sycophancy_sampling_frame.xlsx"

SEARCH_LOG_CSV = "github_sycophancy_search_log.csv"


In [ ]:
# Optional: corpus size estimate (dry run)
#
# Set RUN_DRY_RUN = True to query only the total_count for every repo-term pair
# (one API request each, no pagination, no issue bodies). This reports the exact
# number of raw hits per query before deduplication, at a fraction of the cost
# of full collection, and saves the counts to a CSV for inspection.
# At the standard pacing this takes roughly 1.5 to 2 hours for 3,180 queries.

RUN_DRY_RUN = False

if RUN_DRY_RUN:
    import time
    import requests
    import pandas as pd
    from tqdm import tqdm

    count_rows = []

    for repo in tqdm(REPOS, desc="Repositories"):
        for dimension, terms in SYCOPHANCY_TERMS.items():
            for term, level in terms:
                query = (
                    f'repo:{repo} is:issue "{term}" '
                    f'created:{START_DATE.isoformat()}..{END_DATE.isoformat()}'
                )
                response = requests.get(
                    "https://api.github.com/search/issues",
                    headers=HEADERS,
                    params={"q": query, "per_page": 1}
                )
                if response.status_code == 200:
                    total = response.json().get("total_count")
                else:
                    total = None
                    if response.status_code in [403, 429]:
                        time.sleep(60)

                count_rows.append({
                    "repo": repo,
                    "dimension": dimension,
                    "search_term": term,
                    "total_count": total
                })
                time.sleep(SEARCH_SLEEP_SECONDS)

    counts_df = pd.DataFrame(count_rows)
    counts_df.to_csv("github_sycophancy_dry_run_counts.csv",
                     index=False, encoding="utf-8-sig")

    print(f"Total raw hits (before deduplication): {counts_df['total_count'].sum():,.0f}")
    print("\nTop 20 repo-term pairs by hit count:")
    display(counts_df.sort_values("total_count", ascending=False).head(20))
    print("\nHits by search term (summed over repositories):")
    display(counts_df.groupby("search_term")["total_count"].sum()
            .sort_values(ascending=False).head(30))


In [ ]:
import math
import pandas as pd
from datetime import timedelta
from tqdm import tqdm


def github_get(url, params=None, max_retries=5):
    for attempt in range(max_retries):
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code == 200:
            return response.json()

        if response.status_code in [403, 429]:
            reset_time = response.headers.get("x-ratelimit-reset")
            remaining = response.headers.get("x-ratelimit-remaining")

            if remaining == "0" and reset_time:
                sleep_for = max(int(reset_time) - int(time.time()) + 5, 10)
                print(f"Rate limit reached. Sleeping for {sleep_for} seconds.")
                time.sleep(sleep_for)
            else:
                wait = 60 * (attempt + 1)
                print(f"Secondary rate limit. Sleeping for {wait} seconds.")
                time.sleep(wait)
            continue

        print(f"Request failed: {response.status_code}")
        print(response.text[:300])
        return None

    return None


SEARCH_URL = "https://api.github.com/search/issues"


def item_to_row(item, repo, term, level, dimension):
    return {
        "syco_search_dimension": dimension,
        "search_term": term,
        "search_term_level": level,
        "repo": repo,
        "repo_category": REPO_CATEGORIES.get(repo, ""),
        "issue_number": item.get("number"),
        "title": item.get("title"),
        "body": item.get("body"),
        "issue_author": item.get("user", {}).get("login") if item.get("user") else None,
        "issue_author_association": item.get("author_association"),
        "state": item.get("state"),
        "created_at": item.get("created_at"),
        "updated_at": item.get("updated_at"),
        "closed_at": item.get("closed_at"),
        "comments_count": item.get("comments"),
        "labels": "; ".join([label["name"] for label in item.get("labels", [])]),
        "html_url": item.get("html_url"),
        "comments_url": item.get("comments_url")
    }


def search_issues_window(repo, term, level, dimension,
                         window_start, window_end, search_log):
    """
    Exhaustively retrieve all issues matching a repo-term query within a date
    window. If the window exceeds GitHub's 1,000-result cap, it is split
    recursively into two smaller windows. Every executed query is logged with
    its total_count, retrieved count, and truncation status.
    """
    rows = []
    query = (
        f'repo:{repo} is:issue "{term}" '
        f'created:{window_start.isoformat()}..{window_end.isoformat()}'
    )

    params = {"q": query, "sort": "created", "order": "asc",
              "per_page": PER_PAGE, "page": 1}
    data = github_get(SEARCH_URL, params=params)
    time.sleep(SEARCH_SLEEP_SECONDS)

    if not data or "items" not in data:
        search_log.append({
            "repo": repo, "search_term": term,
            "dimension": dimension, "window_start": window_start.isoformat(),
            "window_end": window_end.isoformat(), "total_count": None,
            "retrieved": 0, "action": "request_failed", "truncated": True
        })
        return rows

    total_count = data.get("total_count", 0)

    # Over the API cap: split the window in half and recurse (if splittable).
    if total_count > API_RESULT_CAP and window_start < window_end:
        mid = window_start + (window_end - window_start) / 2
        search_log.append({
            "repo": repo, "search_term": term,
            "dimension": dimension, "window_start": window_start.isoformat(),
            "window_end": window_end.isoformat(), "total_count": total_count,
            "retrieved": 0, "action": "split", "truncated": False
        })
        rows.extend(search_issues_window(
            repo, term, level, dimension, window_start, mid, search_log))
        rows.extend(search_issues_window(
            repo, term, level, dimension, mid + timedelta(days=1), window_end, search_log))
        return rows

    for item in data["items"]:
        rows.append(item_to_row(item, repo, term, level, dimension))

    pages_needed = min(math.ceil(total_count / PER_PAGE), MAX_PAGES_PER_QUERY)

    for page in range(2, pages_needed + 1):
        params["page"] = page
        data = github_get(SEARCH_URL, params=params)
        time.sleep(SEARCH_SLEEP_SECONDS)

        if not data or "items" not in data or len(data["items"]) == 0:
            break

        for item in data["items"]:
            rows.append(item_to_row(item, repo, term, level, dimension))

    # Truncated only in the unsplittable case: a single-day window over the cap.
    truncated = total_count > API_RESULT_CAP
    near_cap = total_count >= NEAR_CAP_THRESHOLD

    search_log.append({
        "repo": repo, "search_term": term, "term_level": level,
        "dimension": dimension, "window_start": window_start.isoformat(),
        "window_end": window_end.isoformat(), "total_count": total_count,
        "retrieved": len(rows), "action": "collected",
        "truncated": truncated, "near_cap": near_cap
    })

    return rows


all_rows = []
search_log = []

for repo in tqdm(REPOS, desc="Repositories"):
    for dimension, terms in SYCOPHANCY_TERMS.items():
        for term, level in tqdm(terms, desc=repo, leave=False):
            all_rows.extend(search_issues_window(
                repo, term, level, dimension,
                START_DATE, END_DATE, search_log))

search_log_df = pd.DataFrame(search_log)
search_log_df.to_csv(SEARCH_LOG_CSV, index=False, encoding="utf-8-sig")

truncated_queries = search_log_df[search_log_df["truncated"] == True]
near_cap_queries = search_log_df[search_log_df.get("near_cap", False) == True]
print(f"Executed queries logged: {len(search_log_df)}")
print(f"Truncated or failed queries: {len(truncated_queries)}")
print(f"Queries near the 1,000-result cap: {len(near_cap_queries)}")

issues_df = pd.DataFrame(all_rows)

# Duplicates can occur across search terms and across adjacent date windows.
# Before removing them, record every search term, facet, and term type that
# retrieved each issue, so multi-term hits are not lost.
term_map = (
    issues_df
    .groupby(["repo", "issue_number"])
    .agg(
        all_search_terms=("search_term", lambda s: "; ".join(sorted(set(s)))),
        all_search_dimensions=("syco_search_dimension", lambda s: "; ".join(sorted(set(s)))),
        all_search_term_levels=("search_term_level", lambda s: "; ".join(sorted(set(s))))
    )
    .reset_index()
)

issues_df = issues_df.drop_duplicates(subset=["repo", "issue_number"])
issues_df = issues_df.merge(term_map, on=["repo", "issue_number"], how="left")

issues_df.to_csv(CANDIDATE_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(issues_df)} unique candidate issues.")
issues_df.head()


In [ ]:
import re

# If issues_df is still in memory, use it.
# If not, reload from the CSV that was already saved.
try:
    issues_df
    print("Using existing issues_df in memory.")
except NameError:
    issues_df = pd.read_csv(CANDIDATE_CSV)
    print("Reloaded issues_df from CSV.")

# Remove illegal Excel characters
ILLEGAL_CHARACTERS_RE = re.compile(r"[\000-\010]|[\013-\014]|[\016-\037]")

def clean_excel_text(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

issues_df_clean = issues_df.applymap(clean_excel_text)

# Save cleaned Excel version
issues_df_clean.to_excel(CANDIDATE_EXCEL, index=False)

print(f"Saved cleaned Excel file with {len(issues_df_clean)} unique candidate issues.")


In [ ]:
from google.colab import files

files.download(CANDIDATE_EXCEL)
files.download(CANDIDATE_CSV)
files.download(SEARCH_LOG_CSV)


In [ ]:
# Load the cleaned candidate issue dataset
issues_df = pd.read_excel(CANDIDATE_EXCEL)

# Make sure comments_count is numeric
issues_df["comments_count"] = pd.to_numeric(
    issues_df["comments_count"], errors="coerce"
).fillna(0).astype(int)

# All unique candidate issues proceed to comment retrieval and coding.
# No comment-count filter or per-dimension sampling is applied.
issues_df["zero_comments"] = (issues_df["comments_count"] == 0).astype(int)

issues_for_coding = issues_df.copy().reset_index(drop=True)

print("Candidate issues by repository category:")
display(issues_for_coding["repo_category"].value_counts())

print("\nCandidate issues by sycophancy search dimension (first-retrieved):")
display(issues_for_coding["syco_search_dimension"].value_counts())

print(f"Prepared {len(issues_for_coding)} unique candidate issues for comment retrieval.")


In [ ]:
def get_issue_comments(comments_url):
    comments = []
    page = 1

    while True:
        params = {
            "per_page": 100,
            "page": page
        }

        data = github_get(comments_url, params=params)

        if not data or len(data) == 0:
            break

        for comment in data:
            comments.append({
                "comment_id": comment.get("id"),
                "comment_author": comment.get("user", {}).get("login") if comment.get("user") else None,
                "comment_author_association": comment.get("author_association"),
                "comment_created_at": comment.get("created_at"),
                "comment_updated_at": comment.get("updated_at"),
                "comment_body": comment.get("body"),
                "comment_url": comment.get("html_url")
            })

        page += 1
        time.sleep(1)

    return comments


# Comment retrieval is the longest stage. Progress is checkpointed to
# COMMENTS_CSV every CHECKPOINT_EVERY processed issues, and already-retrieved
# issues are skipped on rerun, so an interrupted session resumes without loss.
CHECKPOINT_EVERY = 200

import os

if os.path.exists(COMMENTS_CSV):
    existing_comments_df = pd.read_csv(COMMENTS_CSV)
    all_comments = existing_comments_df.to_dict("records")
    done_issues = set(zip(existing_comments_df["repo"],
                          existing_comments_df["issue_number"]))
    print(f"Resuming: comments for {len(done_issues)} issues already retrieved.")
else:
    all_comments = []
    done_issues = set()

processed_since_checkpoint = 0

for _, row in tqdm(issues_for_coding.iterrows(), total=len(issues_for_coding)):
    comments_url = row["comments_url"]

    # Issues with no comments have nothing to retrieve
    if pd.isna(comments_url) or row["comments_count"] == 0:
        continue

    if (row["repo"], row["issue_number"]) in done_issues:
        continue

    comments = get_issue_comments(comments_url)

    for comment in comments:
        comment.update({
            "repo": row["repo"],
            "repo_category": row["repo_category"],
            "issue_number": row["issue_number"],
            "issue_title": row["title"],
            "issue_url": row["html_url"],
            "syco_search_dimension": row["syco_search_dimension"],
            "search_term": row["search_term"]
        })

        all_comments.append(comment)

    processed_since_checkpoint += 1
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        pd.DataFrame(all_comments).to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")
        processed_since_checkpoint = 0

comments_df = pd.DataFrame(all_comments)

comments_df.to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(comments_df)} comments from {len(issues_for_coding)} candidate issues.")
comments_df.head()


In [ ]:
thread_rows = []

for _, issue in issues_for_coding.iterrows():
    repo = issue["repo"]
    issue_number = issue["issue_number"]

    # Structured thread text with speaker and timestamp markers, in
    # chronological order. The raw comments file remains the authoritative
    # source for comment-level metadata.
    author = issue.get("issue_author", "") or "unknown"
    association = issue.get("issue_author_association", "") or "NONE"
    created = issue.get("created_at", "") or ""

    parts = [
        f"[ISSUE | {author} ({association}) | {created}]",
        f"TITLE: {issue.get('title', '')}",
        "",
        "BODY:",
        str(issue.get("body", "") or "")
    ]

    if len(comments_df) > 0:
        issue_comments = comments_df[
            (comments_df["repo"] == repo) &
            (comments_df["issue_number"] == issue_number)
        ].sort_values("comment_created_at")

        for j, (_, comment) in enumerate(issue_comments.iterrows(), start=1):
            c_author = comment.get("comment_author", "") or "unknown"
            c_association = comment.get("comment_author_association", "") or "NONE"
            c_created = comment.get("comment_created_at", "") or ""
            c_body = str(comment.get("comment_body", "") or "")

            parts.append("")
            parts.append(f"[COMMENT {j} | {c_author} ({c_association}) | {c_created}]")
            parts.append(c_body)

    full_text = "\n".join(parts).strip()

    thread_rows.append({
        "repo": repo,
        "repo_category": issue.get("repo_category", ""),
        "issue_number": issue_number,
        "title": issue.get("title", ""),
        "issue_author": issue.get("issue_author", ""),
        "issue_author_association": issue.get("issue_author_association", ""),
        "state": issue.get("state", ""),
        "created_at": issue.get("created_at", ""),
        "updated_at": issue.get("updated_at", ""),
        "closed_at": issue.get("closed_at", ""),
        "comments_count": issue.get("comments_count", 0),
        "labels": issue.get("labels", ""),
        "html_url": issue.get("html_url", ""),
        "zero_comments": issue.get("zero_comments", ""),
        "syco_search_dimension": issue.get("syco_search_dimension", ""),
        "search_term": issue.get("search_term", ""),
        "search_term_level": issue.get("search_term_level", ""),
        "all_search_terms": issue.get("all_search_terms", ""),
        "all_search_dimensions": issue.get("all_search_dimensions", ""),
        "all_search_term_levels": issue.get("all_search_term_levels", ""),
        "full_text": full_text
    })

threads_df = pd.DataFrame(thread_rows)

threads_df.to_csv(THREADS_CSV, index=False, encoding="utf-8-sig")

print(f"Prepared {len(threads_df)} issue threads.")
threads_df.head()


In [ ]:
threads_df_clean = threads_df.applymap(clean_excel_text)
comments_df_clean = comments_df.applymap(clean_excel_text)

threads_df_clean.to_excel(THREADS_EXCEL, index=False)
comments_df_clean.to_excel(COMMENTS_EXCEL, index=False)

print("Saved cleaned Excel files.")


In [ ]:
# Stage-2 sampling frame: repository caps and dimension floors
#
# The full candidate corpus remains the collection output. This cell adds a
# reproducible, seeded selection flag implementing the stratified design for
# downstream screening and coding:
#   1. every issue from repositories at or under MAX_PER_REPO is selected;
#   2. repositories over the cap are downsampled to MAX_PER_REPO, stratified
#      proportionally by search dimension (seed RANDOM_SEED), so prolific
#      repositories such as anthropics/claude-code cannot dominate the coded set;
#   3. any dimension with fewer than MIN_PER_DIMENSION selected issues
#      (multi-label membership via all_search_dimensions) is topped up from
#      unselected issues where available.
# Selection is recorded as a flag; no rows are dropped.

import numpy as np

frame = threads_df.copy()
frame["selected_for_coding"] = False

repo_counts = frame["repo"].value_counts()

# Step 1: repositories at or under the cap
under_cap = repo_counts[repo_counts <= MAX_PER_REPO].index
frame.loc[frame["repo"].isin(under_cap), "selected_for_coding"] = True

# Step 2: proportional stratified downsampling for over-cap repositories
for repo in repo_counts[repo_counts > MAX_PER_REPO].index:
    sub = frame[frame["repo"] == repo]
    selected_idx = []

    for dim, group in sub.groupby("syco_search_dimension"):
        quota = int(round(len(group) / len(sub) * MAX_PER_REPO))
        quota = min(max(quota, 1), len(group))
        selected_idx.extend(
            group.sample(n=quota, random_state=RANDOM_SEED).index.tolist()
        )

    # Rounding can leave the selection slightly over or under the cap
    if len(selected_idx) > MAX_PER_REPO:
        rng = np.random.default_rng(RANDOM_SEED)
        selected_idx = list(rng.choice(selected_idx, size=MAX_PER_REPO, replace=False))
    elif len(selected_idx) < MAX_PER_REPO:
        remaining = sub.index.difference(selected_idx)
        top_up = sub.loc[remaining].sample(
            n=min(MAX_PER_REPO - len(selected_idx), len(remaining)),
            random_state=RANDOM_SEED
        ).index.tolist()
        selected_idx.extend(top_up)

    frame.loc[selected_idx, "selected_for_coding"] = True

# Step 3: dimension floors (multi-label membership)
for dim in SYCOPHANCY_TERMS.keys():
    member = frame["all_search_dimensions"].fillna("").str.contains(dim, regex=False)
    n_selected = int((member & frame["selected_for_coding"]).sum())

    if n_selected < MIN_PER_DIMENSION:
        pool = frame[member & ~frame["selected_for_coding"]]
        n_needed = min(MIN_PER_DIMENSION - n_selected, len(pool))
        if n_needed > 0:
            top_up = pool.sample(n=n_needed, random_state=RANDOM_SEED)
            frame.loc[top_up.index, "selected_for_coding"] = True

frame_clean = frame.applymap(clean_excel_text)
frame_clean.to_csv(SAMPLING_FRAME_CSV, index=False, encoding="utf-8-sig")
frame_clean.to_excel(SAMPLING_FRAME_EXCEL, index=False)

print(f"Sampling frame saved: {frame['selected_for_coding'].sum()} of {len(frame)} "
      f"threads selected for screening and coding.")

print("\nSelected threads by repository (top 15):")
display(frame[frame["selected_for_coding"]]["repo"].value_counts().head(15))

print("\nSelected threads by dimension (multi-label membership):")
for dim in SYCOPHANCY_TERMS.keys():
    member = frame["all_search_dimensions"].fillna("").str.contains(dim, regex=False)
    print(f"  {dim}: {int((member & frame['selected_for_coding']).sum())}")


In [ ]:
files.download(COMMENTS_EXCEL)
files.download(THREADS_EXCEL)
files.download(SAMPLING_FRAME_EXCEL)
files.download(SAMPLING_FRAME_CSV)


In [ ]:
# Final check: collection and sampling funnel for reporting
threads_check = pd.read_excel(THREADS_EXCEL)
frame_check = pd.read_csv(SAMPLING_FRAME_CSV)
log_check = pd.read_csv(SEARCH_LOG_CSV)

raw_hits = log_check.loc[log_check["action"] == "collected", "retrieved"].sum()

print("Reporting funnel")
print(f"  1. Raw keyword hits (before deduplication): {raw_hits}")
print(f"  2. Unique candidate issues: {len(threads_check)}")
print(f"  3. Zero-comment issues (labelled): {int(threads_check['zero_comments'].sum())}")
print(f"  4. Selected for screening/coding (repo caps + dimension floors): "
      f"{int(frame_check['selected_for_coding'].sum())}")
print("\nSubsequent stages (outside this notebook): false-positive screening, ")
print("multi-label coding, and validated sycophancy case counts.")

print("\nThreads by repository category:")
display(threads_check["repo_category"].value_counts())

print("\nThreads by repository (top 15):")
display(threads_check["repo"].value_counts().head(15))

print("\nThreads by search term level (any matching term):")
for level in ["core", "behavioural", "exploratory"]:
    n = threads_check["all_search_term_levels"].fillna("").str.contains(level).sum()
    print(f"  {level}: {n}")


In [ ]:
print("Final reproducibility settings")
print(f"Repositories: {len(REPOS)}")
print(f"Search terms: {sum(len(v) for v in SYCOPHANCY_TERMS.values())}")
print(f"Collection window (issue created): {START_DATE.isoformat()} to {END_DATE.isoformat()}")
print(f"Search sleep seconds: {SEARCH_SLEEP_SECONDS}")
print(f"Result cap handling: recursive date-window splitting at {API_RESULT_CAP}; "
      f"near-cap flag at {NEAR_CAP_THRESHOLD}")
print(f"Sampling frame: max {MAX_PER_REPO} per repository, "
      f"minimum {MIN_PER_DIMENSION} per dimension, seed {RANDOM_SEED}")
